# Hierarchical Clustering: Latihan Interaktif

Notebook ini berisi latihan hands-on untuk memahami hierarchical (agglomerative) clustering.

Lihat `README.md` untuk teori lengkap.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

from implementation import AgglomerativeClustering, generate_hierarchical_data, plot_dendrogram, plot_clustering_results

np.random.seed(42)
print("✓ Setup complete!")

## Latihan 1: Basic Hierarchical Clustering

In [ ]:
# Generate outlet data
X = generate_hierarchical_data(n_samples=150, random_state=42)

# Fit hierarchical clustering
hc = AgglomerativeClustering(n_clusters=4, linkage='average')
labels = hc.fit_predict(X)

print(f"Clusters found: {hc.n_clusters_}")
for i in range(hc.n_clusters_):
    count = np.sum(labels == i)
    print(f"Cluster {i}: {count} outlets")

# Plot
fig = plot_clustering_results(X, labels, "Hierarchical Clustering (Average Linkage)")
plt.show()

## Latihan 2: Dendrogram Visualization

In [ ]:
# Use smaller dataset for clearer dendrogram
X_small = generate_hierarchical_data(n_samples=50, random_state=42)

# Fit
hc_small = AgglomerativeClustering(n_clusters=3, linkage='average')
hc_small.fit(X_small)

# Plot dendrogram
fig = plot_dendrogram(hc_small, "Dendrogram - Outlet Hierarchy")
plt.show()

print("The dendrogram shows how outlets are merged hierarchically.")
print("Lower merge height = more similar outlets.")

## Latihan 3: Linkage Method Comparison

In [ ]:
linkages = ['single', 'complete', 'average']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, linkage_method in zip(axes, linkages):
    hc = AgglomerativeClustering(n_clusters=4, linkage=linkage_method)
    labels = hc.fit_predict(X)
    
    colors = plt.cm.Spectral(np.linspace(0, 1, 4))
    for i, color in enumerate(colors):
        mask = labels == i
        ax.scatter(X[mask, 0], X[mask, 1], c=[color], s=60, alpha=0.7)
    
    ax.set_title(f'{linkage_method.capitalize()} Linkage', fontweight='bold', fontsize=13)
    ax.set_xlabel('X Coordinate')
    ax.set_ylabel('Y Coordinate')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Observations:")
print("- Single: Can create elongated/chain-like clusters")
print("- Complete: Creates compact, spherical clusters")
print("- Average: Balanced trade-off")

## Latihan 4: Cutting Dendrogram at Different Levels

In [ ]:
# Create linkage matrix
Z = linkage(X_small, method='average')

# Cut at different thresholds
thresholds = [2.0, 3.0, 4.0]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

# Plot dendrogram with cut lines
ax = axes[0]
dendrogram(Z, ax=ax, color_threshold=3.0)
for threshold, color in zip(thresholds, ['red', 'green', 'blue']):
    ax.axhline(y=threshold, color=color, linestyle='--', linewidth=2, label=f'Cut at {threshold}')
ax.set_title('Dendrogram with Cut Levels', fontweight='bold')
ax.set_xlabel('Sample Index')
ax.set_ylabel('Distance')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Plot clustering results for each threshold
for idx, threshold in enumerate(thresholds, start=1):
    labels = fcluster(Z, threshold, criterion='distance')
    n_clusters = len(np.unique(labels))
    
    ax = axes[idx]
    colors = plt.cm.Spectral(np.linspace(0, 1, n_clusters))
    for i in range(1, n_clusters + 1):
        mask = labels == i
        ax.scatter(X_small[mask, 0], X_small[mask, 1], c=[colors[i-1]], s=80, alpha=0.7)
    
    ax.set_title(f'Cut at {threshold} → {n_clusters} clusters', fontweight='bold')
    ax.set_xlabel('X Coordinate')
    ax.set_ylabel('Y Coordinate')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("One hierarchy, multiple cluster choices!")

## Latihan 5: Choosing Optimal Number of Clusters

In [ ]:
from sklearn.metrics import silhouette_score

# Test different n_clusters
n_range = range(2, 11)
silhouette_scores = []

for n in n_range:
    hc = AgglomerativeClustering(n_clusters=n, linkage='average')
    labels = hc.fit_predict(X)
    score = silhouette_score(X, labels)
    silhouette_scores.append(score)
    print(f"n_clusters={n:2d}: Silhouette={score:.4f}")

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(list(n_range), silhouette_scores, marker='o', linewidth=2, markersize=8)

best_n = list(n_range)[np.argmax(silhouette_scores)]
best_score = max(silhouette_scores)
ax.axvline(x=best_n, color='red', linestyle='--', linewidth=2, label=f'Best: {best_n}')
ax.scatter([best_n], [best_score], color='red', s=200, zorder=5, edgecolors='black', linewidth=2)

ax.set_xlabel('Number of Clusters', fontsize=12)
ax.set_ylabel('Silhouette Score', fontsize=12)
ax.set_title('Finding Optimal Number of Clusters', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(n_range)

plt.show()

print(f"\nOptimal: {best_n} clusters (highest silhouette score)")

## Latihan 6: Hierarchical vs K-Means

In [ ]:
from sklearn.cluster import KMeans

# Hierarchical
hc = AgglomerativeClustering(n_clusters=4, linkage='average')
hc_labels = hc.fit_predict(X)
hc_silhouette = silhouette_score(X, hc_labels)

# K-Means
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
km_labels = kmeans.fit_predict(X)
km_silhouette = silhouette_score(X, km_labels)

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Hierarchical
ax = axes[0]
colors = plt.cm.Spectral(np.linspace(0, 1, 4))
for i, color in enumerate(colors):
    mask = hc_labels == i
    ax.scatter(X[mask, 0], X[mask, 1], c=[color], s=80, alpha=0.7)
ax.set_title(f'Hierarchical\nSilhouette: {hc_silhouette:.3f}', fontweight='bold', fontsize=13)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.grid(True, alpha=0.3)

# K-Means
ax = axes[1]
for i, color in enumerate(colors):
    mask = km_labels == i
    ax.scatter(X[mask, 0], X[mask, 1], c=[color], s=80, alpha=0.7)
ax.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
          c='red', marker='X', s=300, edgecolors='black', linewidth=2, label='Centroids')
ax.set_title(f'K-Means\nSilhouette: {km_silhouette:.3f}', fontweight='bold', fontsize=13)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Both methods work well on this dataset.")
print("Hierarchical advantage: dendrogram shows hierarchy, no need to specify k upfront.")
print("K-Means advantage: faster, scales better to large datasets.")